In [1]:
# Install any required libraries (run once)
%pip install pandas numpy scikit-learn imbalanced-learn matplotlib seaborn --quiet
%pip install --quiet nltk

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import string
import nltk
from nltk.corpus import stopwords

In [9]:
# Define correct file paths
fake_path = "Data/Fake.csv"
true_path = "Data/True.csv"


# Load both CSVs
fake_df = pd.read_csv(fake_path)
true_df = pd.read_csv(true_path)

In [10]:
# Add labels: 1 for Fake, 0 for True
fake_df["label"] = 1
true_df["label"] = 0

# Combine both into one DataFrame
news_df = pd.concat([fake_df, true_df], axis=0).reset_index(drop=True)

# Display shape and first few rows
print("Shape of Fake News:", fake_df.shape)
print("Shape of True News:", true_df.shape)
print("Combined Dataset Shape:", news_df.shape)
news_df.head()

Shape of Fake News: (23481, 5)
Shape of True News: (21417, 5)
Combined Dataset Shape: (44898, 5)


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1


In [11]:
news_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   label    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


In [12]:
# Missing values
print(news_df.isnull().sum())

title      0
text       0
subject    0
date       0
label      0
dtype: int64


In [13]:
def remove_punct(text):
    return "".join([char for char in text if char not in string.punctuation])

news_df['text_clean'] = news_df['text'].apply(lambda x: remove_punct(x.lower()))
news_df.head()

,title,text,subject,date,label,text_clean
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1,donald trump just couldn t wish all americans ...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1,house intelligence committee chairman devin nu...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1,on friday it was revealed that former milwauke...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1,on christmas day donald trump announced that h...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1,pope francis used his annual christmas day mes...


In [14]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

# Download all required NLTK datasets
nltk.download('stopwords')   # list of English stopwords
nltk.download('punkt')       # for tokenization
nltk.download('punkt_tab')   # for updated punkt tokenizer tables
nltk.download('wordnet')     # for lemmatization
nltk.download('omw-1.4')     # WordNet linguistic data (used with lemmatizer)



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [15]:
news_df['text_tokens'] = news_df['text_clean'].apply(word_tokenize)
news_df.head()

,title,text,subject,date,label,text_clean,text_tokens
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1,donald trump just couldn t wish all americans ...,"[donald, trump, just, couldn, t, wish, all, am..."
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1,house intelligence committee chairman devin nu...,"[house, intelligence, committee, chairman, dev..."
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1,on friday it was revealed that former milwauke...,"[on, friday, it, was, revealed, that, former, ..."
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1,on christmas day donald trump announced that h...,"[on, christmas, day, donald, trump, announced,..."
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1,pope francis used his annual christmas day mes...,"[pope, francis, used, his, annual, christmas, ..."


In [16]:

stopwords_En = stopwords.words('english')

news_df['text_tokens'] = news_df['text_tokens'].apply(lambda tokens: [word for word in tokens if word not in stopwords_En])
news_df.head()

,title,text,subject,date,label,text_clean,text_tokens
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1,donald trump just couldn t wish all americans ...,"[donald, trump, wish, americans, happy, new, y..."
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1,house intelligence committee chairman devin nu...,"[house, intelligence, committee, chairman, dev..."
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1,on friday it was revealed that former milwauke...,"[friday, revealed, former, milwaukee, sheriff,..."
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1,on christmas day donald trump announced that h...,"[christmas, day, donald, trump, announced, wou..."
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1,pope francis used his annual christmas day mes...,"[pope, francis, used, annual, christmas, day, ..."


In [17]:
wn = WordNetLemmatizer()

news_df['text_tokens'] = news_df['text_tokens'].apply(lambda tokens: [wn.lemmatize(word) for word in tokens])
news_df.head()

,title,text,subject,date,label,text_clean,text_tokens
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1,donald trump just couldn t wish all americans ...,"[donald, trump, wish, american, happy, new, ye..."
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1,house intelligence committee chairman devin nu...,"[house, intelligence, committee, chairman, dev..."
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1,on friday it was revealed that former milwauke...,"[friday, revealed, former, milwaukee, sheriff,..."
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1,on christmas day donald trump announced that h...,"[christmas, day, donald, trump, announced, wou..."
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1,pope francis used his annual christmas day mes...,"[pope, francis, used, annual, christmas, day, ..."


In [18]:
# Initialize lemmatizer
wn = WordNetLemmatizer()

# Lemmatization (list of tokens → lemmatized tokens)
news_df['text_tokens'] = news_df['text_tokens'].apply(
    lambda tokens: [wn.lemmatize(word) for word in tokens]
)

# Join tokens back into a single string for TF-IDF
news_df['text_final'] = news_df['text_tokens'].apply(lambda tokens: ' '.join(tokens))

In [19]:
news_df.head()

,title,text,subject,date,label,text_clean,text_tokens,text_final
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",1,donald trump just couldn t wish all americans ...,"[donald, trump, wish, american, happy, new, ye...",donald trump wish american happy new year leav...
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",1,house intelligence committee chairman devin nu...,"[house, intelligence, committee, chairman, dev...",house intelligence committee chairman devin nu...
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",1,on friday it was revealed that former milwauke...,"[friday, revealed, former, milwaukee, sheriff,...",friday revealed former milwaukee sheriff david...
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",1,on christmas day donald trump announced that h...,"[christmas, day, donald, trump, announced, wou...",christmas day donald trump announced would bac...
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",1,pope francis used his annual christmas day mes...,"[pope, francis, used, annual, christmas, day, ...",pope francis used annual christmas day message...


In [22]:
# ============================================================
# 0️⃣ Install libraries (run once per Colab session)
# ============================================================

%pip install -U "transformers" "datasets" "peft" "accelerate" "torch" --quiet
%pip install tf-keras

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [23]:
# ============================================================
# 1️⃣ Imports
# ============================================================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import seaborn as sns
import matplotlib.pyplot as plt

import torch
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments
)

from peft import LoraConfig, get_peft_model, TaskType

# ============================================================
# 2️⃣ Prepare text and labels (use your preprocessed text_final)
# ============================================================
# X = list of texts, y = list of labels (0 = true, 1 = fake)
X = news_df['text_final'].tolist()
y = news_df['label'].tolist()

# Train/test split (same as before)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [24]:
# ============================================================
# 3️⃣ Tokenize using RoBERTa tokenizer
# ============================================================
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def tokenize(batch):
    # batch["text"] is a list of strings
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

# HuggingFace Datasets from Python lists
train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_dataset  = Dataset.from_dict({"text": test_texts,  "label": test_labels})

# Apply tokenizer
train_dataset = train_dataset.map(tokenize, batched=True, batch_size=len(train_dataset))
test_dataset  = test_dataset.map(tokenize,  batched=True, batch_size=len(test_dataset))

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch",  columns=["input_ids", "attention_mask", "label"])

# ============================================================
# 4️⃣ Load base RoBERTa model
# ============================================================
base_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Map:   0%|          | 0/35918 [00:00<?, ? examples/s]

Map:   0%|          | 0/8980 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
%pip install -U transformers


Note: you may need to restart the kernel to use updated packages.


In [26]:
import os
os.environ["WANDB_DISABLED"] = "true"



# ============================================================
# 5️⃣ Wrap model with LoRA (PEFT)
# ============================================================
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # sequence classification
    r=8,                          # rank of LoRA matrices (low-rank size)
    lora_alpha=16,                # scaling factor
    lora_dropout=0.1,             # dropout on LoRA layers
    bias="none",
    target_modules=["query", "value"]  # which submodules to adapt in attention
)

model = get_peft_model(base_model, lora_config)

# Optional: see how many params are trainable vs frozen
model.print_trainable_parameters()

# ============================================================
# ⚙️ 6️⃣ Training setup — limit total steps to 800
# ============================================================
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

# ============================================================
# ⚙️ Training setup — compatible with older Transformers versions
# ============================================================
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

# ============================================================
# ⚙️ Training setup — minimal and version-compatible
# ============================================================
training_args = TrainingArguments(
    output_dir="./roberta_lora_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,     # still required
    max_steps=800,          # ✅ stops after 800 steps
    save_steps=200,         # save every 200 steps
    logging_dir="./logs",
    logging_steps=50
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# ============================================================
# 🚀 Train RoBERTa + LoRA — limited to 800 optimizer steps
# ============================================================
trainer.train()


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


/usr/local/lib/python3.11/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.694200
100,0.604800
150,0.288300
200,0.133800
250,0.132900
300,0.164400
350,0.141300
400,0.081000
450,0.049700
500,0.057500


TrainOutput(global_step=800, training_loss=0.1680518315732479, metrics={'train_runtime': 3263.5262, 'train_samples_per_second': 1.961, 'train_steps_per_second': 0.245, 'total_flos': 850675354828800.0, 'train_loss': 0.1680518315732479, 'epoch': 0.17817371937639198})

In [ ]:
# ============================================================
# 8️⃣ Evaluate on test set
# ============================================================
preds_output = trainer.predict(test_dataset)
y_pred = preds_output.predictions.argmax(-1)
y_true = test_labels

print("Accuracy:", accuracy_score(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred))

# ============================================================
# 9️⃣ Confusion Matrix Visualization
# ============================================================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Predicted True (0)", "Predicted Fake (1)"],
    yticklabels=["Actual True (0)", "Actual Fake (1)"]
)
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.title("RoBERTa + LoRA Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

# ===============================
#  Predictions
# ===============================
preds_output = trainer.predict(test_dataset)
y_pred = preds_output.predictions.argmax(-1)
y_true = test_labels

# Raw probabilities for AUC-ROC
y_prob = preds_output.predictions[:, 1]  # column 1 = probability of class "1" (fake)

# ===============================
#  Compute Metrics
# ===============================
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)
roc_auc   = roc_auc_score(y_true, y_prob)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"AUC-ROC  : {roc_auc:.4f}\n")

# Detailed report
print("Classification Report:\n", classification_report(y_true, y_pred, digits=4))

# ===============================
#


In [ ]:
#🧠 Install explainability libraries
%pip install shap lime --quiet


In [ ]:
# 🧠 Set up the SHAP explainer

import shap
import torch
from transformers import pipeline

# ===============================
# 1️⃣ Create a text classification pipeline
# ===============================
# Uses your fine-tuned LoRA + RoBERTa model
explainer_pipeline = pipeline(
    "text-classification",
    model=model,       # your fine-tuned model
    tokenizer=tokenizer,
    return_all_scores=True,
    truncation=True,
    max_length=256
)

# ===============================
# 2️⃣ Select a few example texts to interpret
# ===============================
sample_texts = [
    news_df['text_final'].iloc[0],
    news_df['text_final'].iloc[100],
    news_df['text_final'].iloc[200]
]

# ===============================
# 3️⃣ Build the SHAP explainer
# ===============================
explainer = shap.Explainer(explainer_pipeline)

# ===============================
# 4️⃣ Compute SHAP values for sample texts
# ===============================
shap_values = explainer(sample_texts)

# ===============================
# 5️⃣ Visualize the token-level importance
# ===============================
shap.plots.text(shap_values[0])
